# 01 · Data Cleaning: Brightbite Delivery Times

**Project:** Why are Brightbite's deliveries late?
Brightbite is a fictional delivery marketplace; the data is real DoorDash delivery data from Kaggle ([DoorDash ETA Prediction](https://www.kaggle.com/datasets/dharun4772/doordash-eta-prediction)).

**Goal of this notebook:** turn the raw delivery file into one clean, documented table, `deliveries_clean`, that the EDA and modeling notebooks can trust.

**Target variable:** `actual_minutes` = `actual_delivery_time` − `created_at`. This is the full time from the moment a customer places an order to the moment it is delivered: the restaurant receiving the order, food prep, Dasher assignment and travel to the store, pickup, and the drive to the customer. It is not drive time alone.

**Inputs and outputs**

| | File |
| --- | --- |
| Raw data | `data/raw/historical_data.csv` |
| Feature build | `sql/01_build_deliveries.sql` → table `deliveries` |
| Cleaning rules | `sql/02_clean_deliveries.sql` → table `deliveries_clean` |
| Documentation | `docs/cleaning_log.md`, `docs/data_dictionary.md` |

**Contents**
1. Setup
2. First look at the raw data
3. Load into DuckDB and build features
4. Validate the load
5. Data quality checks
6. Missing values
7. Outliers and data errors
8. Build the clean table
9. Findings and next steps

## 1. Setup

In [45]:
import pandas as pd
import duckdb

pd.set_option("display.max_columns", None)

RAW_PATH = "../data/raw/historical_data.csv"
DB_PATH = "../data/processed/brightbite.duckdb"

## 2. First look at the raw data

A quick pandas pass to see the shape, data types and missing values before any transformation.

In [46]:
df = pd.read_csv(RAW_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
print("Data types:")
print(df.dtypes.to_string())

Dataset shape: 197,428 rows × 16 columns

Data types:
market_id                                       float64
created_at                                          str
actual_delivery_time                                str
store_id                                          int64
store_primary_category                              str
order_protocol                                  float64
total_items                                       int64
subtotal                                          int64
num_distinct_items                                int64
min_item_price                                    int64
max_item_price                                    int64
total_onshift_dashers                           float64
total_busy_dashers                              float64
total_outstanding_orders                        float64
estimated_order_place_duration                    int64
estimated_store_to_consumer_driving_duration    float64


In [47]:
missing = df.isna().sum()
missing_table = pd.DataFrame({
    "missing": missing,
    "pct_of_rows": (missing / len(df) * 100).round(2),
}).query("missing > 0").sort_values("missing", ascending=False)
missing_table

,missing,pct_of_rows
total_busy_dashers,16262,8.24
total_outstanding_orders,16262,8.24
total_onshift_dashers,16262,8.24
store_primary_category,4760,2.41
order_protocol,995,0.50
market_id,987,0.50
estimated_store_to_consumer_driving_duration,526,0.27
actual_delivery_time,7,0.00


In [48]:
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


**Observations**

- There are **197,428 deliveries** and 16 columns.
- `created_at` and `actual_delivery_time` load as text, not timestamps. They need converting before the target (total delivery minutes) can be computed.
- Money is stored in **cents**: a `subtotal` of 3441 means $34.41.
- Some store categories are missing but recoverable. Store 5477 appears as `mexican` in one row and `NaN` in others, so a missing category can be filled with the store's most common category.
- One row shows 2 busy Dashers but only 1 on shift, which looks impossible. How often this happens needs counting before deciding what to do.
- Timestamps are in UTC. The data dictionary states every region is in **US/Pacific**, and orders cluster around 01:00–04:00 UTC, which is dinnertime on the West Coast.
- Three Dasher load columns are missing the **same number of rows (16,262)**, which points to one logging gap rather than random noise.

## 3. Load into DuckDB and build features

`sql/01_build_deliveries.sql` loads the CSV and builds one row per delivery with the target and new features:

| Feature | Definition |
| --- | --- |
| `actual_minutes` | Total minutes from order placed to delivered (target) |
| `created_at_pt`, `order_hour_pt`, `order_dow_pt` | Order time converted to US/Pacific (UTC − 8) |
| `busy_ratio` | Busy Dashers ÷ on-shift Dashers; near 1 means the market is saturated |
| `orders_per_dasher` | Outstanding orders ÷ on-shift Dashers; backlog pressure |
| `est_nonprep_minutes` | DoorDash's estimates for order placement + drive time, in minutes |
| `market`, `protocol`, `store_category` | Raw values, with gaps filled from the store's most common value |
| `has_load_data` | 1 if Dasher load counts are present |

**Data issue found while loading:** the raw CSV writes missing values as the text `NA`. Pandas treats that as missing automatically, but DuckDB read the affected numeric columns as text, which broke the division in `busy_ratio`. Fixed by loading with `nullstr = 'NA'` and converting numeric columns with `TRY_CAST`.

In [49]:
con = duckdb.connect(DB_PATH)
con.execute(open("../sql/01_build_deliveries.sql").read())
print(con.execute("SELECT COUNT(*) AS rows FROM deliveries").fetchone())

(197428,)


## 4. Validate the load

Null counts in DuckDB should match pandas exactly. If they don't, the `NA` handling has failed somewhere.

In [50]:
duck_nulls = con.execute('''
    SELECT
        COUNT(*) - COUNT(market_id)              AS market_id,
        COUNT(*) - COUNT(delivered_at_utc)       AS actual_delivery_time,
        COUNT(*) - COUNT(store_primary_category) AS store_primary_category,
        COUNT(*) - COUNT(order_protocol)         AS order_protocol,
        COUNT(*) - COUNT(onshift)                AS total_onshift_dashers,
        COUNT(*) - COUNT(busy)                   AS total_busy_dashers,
        COUNT(*) - COUNT(outstanding)            AS total_outstanding_orders,
        COUNT(*) - COUNT(est_drive_sec)          AS estimated_store_to_consumer_driving_duration
    FROM deliveries
''').df().T[0]

comparison = pd.DataFrame({
    "pandas": df.isna().sum()[duck_nulls.index],
    "duckdb": duck_nulls.astype(int),
})
comparison["match"] = comparison["pandas"] == comparison["duckdb"]
print(comparison)
assert comparison["match"].all(), "DuckDB null counts do not match pandas"

                                              pandas  duckdb  match
market_id                                        987     987   True
actual_delivery_time                               7       7   True
store_primary_category                          4760    4760   True
order_protocol                                   995     995   True
total_onshift_dashers                          16262   16262   True
total_busy_dashers                             16262   16262   True
total_outstanding_orders                       16262   16262   True
estimated_store_to_consumer_driving_duration     526     526   True


**Result:** null counts match pandas for every column, so the DuckDB table is a faithful copy of the raw file.

## 5. Data quality checks

One query to surface date range, impossible values and the shape of the delivery-time tail. These numbers set the cleaning rules.

In [51]:
checks = con.execute('''
    SELECT
        COUNT(*)                                                       AS rows,
        MIN(created_at_utc)                                            AS first_order,
        MAX(created_at_utc)                                            AS last_order,
        SUM(CASE WHEN actual_minutes IS NULL THEN 1 ELSE 0 END)        AS no_delivery_time,
        SUM(CASE WHEN busy > onshift THEN 1 ELSE 0 END)                AS busy_gt_onshift,
        SUM(CASE WHEN onshift < 0 OR busy < 0 OR outstanding < 0
                 THEN 1 ELSE 0 END)                                    AS negative_counts,
        ROUND(QUANTILE_CONT(actual_minutes, 0.001), 1)                 AS p0_1_minutes,
        ROUND(QUANTILE_CONT(actual_minutes, 0.01), 1)                  AS p1_minutes,
        ROUND(QUANTILE_CONT(actual_minutes, 0.5), 1)                   AS median_minutes,
        ROUND(QUANTILE_CONT(actual_minutes, 0.99), 1)                  AS p99_minutes,
        ROUND(QUANTILE_CONT(actual_minutes, 0.999), 1)                 AS p99_9_minutes,
        ROUND(MAX(actual_minutes), 1)                                  AS max_minutes
    FROM deliveries
''').df().T
checks.columns = ["value"]
checks

,value
rows,197428
first_order,2014-10-19 05:24:15
last_order,2015-02-18 06:00:44
no_delivery_time,7.0
busy_gt_onshift,40394.0
negative_counts,81.0
p0_1_minutes,14.1
p1_minutes,19.2
median_minutes,44.3
p99_minutes,107.9


**Observations**

- The last order is **2015-02-18**, before daylight saving began on March 8, 2015, so the UTC − 8 conversion is exact for every row.
- The first order is **2014-10-19**, about three months before the rest of the data. This needs a closer look (Section 7).
- The longest delivery is **141,948 minutes (about 98.6 days)**, which is clearly a data error.
- **40,394 rows** have more busy Dashers than on-shift Dashers, and **81 rows** have negative Dasher counts.

## 6. Missing values

| Column | Missing | Share | Decision | Reason |
| --- | --- | --- | --- | --- |
| `actual_delivery_time` | 7 | <0.01% | Drop | No delivery time means no target to predict |
| `estimated_store_to_consumer_driving_duration` | 526 | 0.3% | Drop | Small share, and a key input to the ETA baseline |
| `market_id` | 987 | 0.5% | Fill from store | A restaurant sits in one market |
| `order_protocol` | 995 | 0.5% | Fill from store | A property of how the store receives orders |
| `store_primary_category` | 4,760 | 2.4% | Fill from store; else `unknown` | Same store, same cuisine |
| Dasher load columns (3) | 16,262 | 8.2% | Test before deciding | Dropping 8% of rows is costly; check whether the gap is related to delivery time |

### 6a. How much did the store-level fill recover?

In [52]:
con.execute('''
    SELECT COUNT(*) - COUNT(market)   AS market_still_missing,
           COUNT(*) - COUNT(protocol) AS protocol_still_missing,
           SUM(CASE WHEN store_category = 'unknown' THEN 1 ELSE 0 END) AS category_still_unknown
    FROM deliveries
''').df()

,market_still_missing,protocol_still_missing,category_still_unknown
0,3,3,867.0


Missing market and protocol fell from about 990 to **3** each (99.7% recovered), and missing categories fell from 4,760 to **867** (82% recovered). The 3 rows are dropped; the 867 stay as `unknown`.

### 6b. Are rows missing Dasher load data different?

In [53]:
con.execute('''
    SELECT has_load_data,
           COUNT(*)                          AS orders,
           ROUND(AVG(actual_minutes), 1)     AS avg_minutes,
           ROUND(MEDIAN(actual_minutes), 1)  AS median_minutes,
           MIN(created_at_utc)               AS first_order,
           MAX(created_at_utc)               AS last_order
    FROM deliveries
    WHERE actual_minutes IS NOT NULL
    GROUP BY has_load_data
    ORDER BY has_load_data
''').df()

,has_load_data,orders,avg_minutes,median_minutes,first_order,last_order
0,0,16262,56.3,43.9,2014-10-19 05:24:15,2015-02-18 05:59:23
1,1,181159,47.8,44.4,2015-01-21 15:22:03,2015-02-18 06:00:44


The medians are nearly identical (43.9 vs. 44.4 minutes), but the averages differ by 8.5 minutes (56.3 vs. 47.8). When the median matches but the mean jumps, a few extreme values are pulling the mean up, so the flagged group probably isn't slower overall. The flagged group also contains the 2014-10-19 orphan order. Both points are tested after outlier review in Section 7.

**Reconciliation:** 16,262 + 181,159 = 197,421, which equals 197,428 minus the 7 rows with no delivery time.

## 7. Outliers and data errors

In [54]:
con.execute('''
    SELECT
        SUM(CASE WHEN created_at_utc < '2015-01-21' THEN 1 ELSE 0 END) AS orders_before_jan21,
        SUM(CASE WHEN actual_minutes > 180 THEN 1 ELSE 0 END)          AS over_3_hours,
        SUM(CASE WHEN actual_minutes < 5 THEN 1 ELSE 0 END)            AS under_5_min,
        ROUND(MAX(actual_minutes), 1)                                  AS longest_minutes
    FROM deliveries
''').df()

,orders_before_jan21,over_3_hours,under_5_min,longest_minutes
0,1.0,138.0,3.0,141947.7


### 7a. Are deliveries under 5 minutes possible?

`actual_minutes` covers the whole order, not just the drive. If DoorDash's own drive-time estimate is longer than the entire recorded delivery, the record can't be right.

In [55]:
con.execute('''
    SELECT delivery_id,
           ROUND(actual_minutes, 1)        AS total_minutes,
           ROUND(est_drive_sec / 60.0, 1)  AS est_drive_minutes,
           ROUND(est_place_sec / 60.0, 1)  AS est_order_place_minutes,
           store_category
    FROM deliveries
    WHERE actual_minutes < 5
''').df()

,delivery_id,total_minutes,est_drive_minutes,est_order_place_minutes,store_category
0,863,3.7,3.6,4.2,sandwich
1,30563,5.0,7.0,4.2,mexican
2,63295,1.7,4.6,4.2,mexican


### 7b. Re-test the missing load data without the extreme tail

In [56]:
con.execute('''
    SELECT has_load_data,
           COUNT(*)                          AS orders,
           ROUND(AVG(actual_minutes), 1)     AS avg_minutes,
           ROUND(MEDIAN(actual_minutes), 1)  AS median_minutes
    FROM deliveries
    WHERE actual_minutes BETWEEN 5 AND 180
      AND created_at_utc >= '2015-01-21'
    GROUP BY has_load_data
    ORDER BY has_load_data
''').df()

,has_load_data,orders,avg_minutes,median_minutes
0,0,16248,47.4,43.9
1,1,181032,47.6,44.4


**Result:** after removing the tail, the two groups are almost identical (47.4 vs. 47.6 minutes average; 43.9 vs. 44.4 median). The earlier 8.5-minute gap came from one 98.6-day delivery in the flagged group. Rows missing load data are therefore **kept and flagged**, not dropped, which preserves 8% of the dataset.

**Decisions**

| Finding | Rows | Decision | Reason |
| --- | --- | --- | --- |
| Orphan order before 2015-01-21 | 1 | Drop | Three months before the rest; would distort a time-based train/test split |
| Deliveries over 180 minutes | 138 (0.07%) | Drop | Data errors or abandoned orders, not something an ETA should predict |
| Deliveries under 5 minutes | 3 | Drop | Shorter than food prep alone; physically implausible |
| Missing market or protocol after fill | 3 | Drop | Cannot be recovered |
| Negative Dasher counts | 81 | Set to null and flag | The count is wrong, but the order itself is valid |
| Busy Dashers > on-shift Dashers | 40,394 (20%) | Keep and flag | Too common to be an error. Most likely Dashers finishing deliveries after their shift ends, so a ratio above 1 signals an overloaded market. Tested in EDA |

## 8. Build the clean table

`sql/02_clean_deliveries.sql` applies every rule above and adds the `busy_exceeds_onshift` flag.

In [57]:
con.execute(open("../sql/02_clean_deliveries.sql").read())

### 8a. Cleaning log: rules applied in order

In [58]:
rules = [
    ("Drop rows with no actual delivery time",       "actual_minutes IS NOT NULL"),
    ("Drop rows with no estimated driving duration", "est_drive_sec IS NOT NULL"),
    ("Drop rows with no market/protocol after fill", "market IS NOT NULL AND protocol IS NOT NULL"),
    ("Drop orphan order before 2015-01-21",          "created_at_utc >= '2015-01-21'"),
    ("Drop deliveries under 5 or over 180 minutes",  "actual_minutes BETWEEN 5 AND 180"),
]

log, conds = [], []
prev = con.execute("SELECT COUNT(*) FROM deliveries").fetchone()[0]
log.append(("Raw load", 0, prev))
for name, cond in rules:
    conds.append(cond)
    n = con.execute(f"SELECT COUNT(*) FROM deliveries WHERE {' AND '.join(conds)}").fetchone()[0]
    log.append((name, prev - n, n))
    prev = n

cleaning_log = pd.DataFrame(log, columns=["rule", "rows_removed", "rows_remaining"])
cleaning_log

,rule,rows_removed,rows_remaining
0,Raw load,0,197428
1,Drop rows with no actual delivery time,7,197421
2,Drop rows with no estimated driving duration,526,196895
3,Drop rows with no market/protocol after fill,3,196892
4,Drop orphan order before 2015-01-21,1,196891
5,Drop deliveries under 5 or over 180 minutes,140,196751


### 8b. Final table and flags

In [59]:
final = con.execute('''
    SELECT COUNT(*)                       AS final_rows,
           SUM(1 - has_load_data)         AS flagged_no_load_data,
           SUM(busy_exceeds_onshift)      AS flagged_busy_gt_onshift
    FROM deliveries_clean
''').df()

raw_rows = cleaning_log.loc[0, "rows_remaining"]
final["pct_of_raw_kept"] = round(final["final_rows"] / raw_rows * 100, 2)

# The rules log and the SQL must agree
assert final.loc[0, "final_rows"] == cleaning_log["rows_remaining"].iloc[-1]
final

,final_rows,flagged_no_load_data,flagged_busy_gt_onshift,pct_of_raw_kept
0,196751,16283.0,40256.0,99.66


## 9. Findings and next steps

**Result:** `deliveries_clean` holds **196,751 deliveries (99.7% of the raw file; 677 rows removed)**.

| Step | Rule | Removed | Remaining |
| --- | --- | --- | --- |
| 0 | Raw load | — | 197,428 |
| 1 | No actual delivery time | 7 | 197,421 |
| 2 | No estimated driving duration | 526 | 196,895 |
| 3 | No market/protocol after store fill | 3 | 196,892 |
| 4 | Orphan order before 2015-01-21 | 1 | 196,891 |
| 5 | Under 5 or over 180 minutes | 140 | 196,751 |

Step 5 removes 140 rather than 138 + 3 = 141 because one extreme row was already removed by an earlier rule.

**Key findings**

1. **Nulls were hidden as text.** The raw file encodes missing values as `NA`, which made DuckDB read numbers as text. Handled at load and validated against pandas.
2. **Store-level fill recovered most gaps.** 99.7% of missing markets and protocols and 82% of missing categories were recovered from each store's other orders.
3. **Missing Dasher data is not tied to delivery time.** Once one 98.6-day outlier is removed, orders with and without load data average 47.4 and 47.6 minutes. Keeping and flagging them saved 16,283 rows (8% of the data).
4. **Busy > on-shift is a pattern, not an error.** It affects 40,256 clean rows (20%) and likely reflects Dashers finishing orders after their shift. It is kept as a flag and tested as a signal of market overload.
5. **Time zone is exact.** All orders fall before daylight saving began, so local time = UTC − 8 for every row.

**Limitations**

- DoorDash added noise to the data, so individual values may be perturbed.
- Markets are anonymous IDs, so findings can't be tied to named cities.
- `busy_exceeds_onshift` is an interpretation of how the data was recorded, not a documented fact.

**Next:** `02_eda.ipynb` covers the delivery-time distribution, delivery time by market and hour, and whether Dasher load drives slow deliveries.

In [60]:
# Release the database file so the next notebook can open it
con.close()